# LeetCode 2186 — Minimum Number of Steps to Make Two Strings Anagram II bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-steps-to-make-two-strings-anagram-ii/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Counting

Skeleton LeetCode:

```python
class Solution:
    def minSteps(self, s: str, t: str) -> int:
        ...
```

Mỗi bước được **thêm** một ký tự bất kỳ vào `s` hoặc `t`; cần ít bước nhất để hai xâu thành **đảo chữ** (anagram) của nhau. Vì chỉ được thêm (không xóa, không sửa), với mỗi chữ cái `c` ta phải bù phần chênh lệch `|cnt_s[c] − cnt_t[c]|`. Notebook đi từ **mô phỏng thêm từng ký tự** → **`Counter` hai chiều** → **mảng 26 ô một lượt** → **sắp xếp + trộn hai con trỏ** → **công thức qua phần giao `len(s) + len(t) − 2·|s ∩ t|`**.

## 1. Mục tiêu học

1. Hiểu "đảo chữ" ⇔ **cùng đa tập ký tự** (cùng số lần xuất hiện của mỗi chữ cái).
2. Chứng minh đáp án là `Σ_c |cnt_s[c] − cnt_t[c]|` (cận dưới + đạt được).
3. Cài đặt bằng `Counter`, mảng 26 ô (một mảng chênh lệch), và qua **giao đa tập** `Counter & Counter`.
4. Phân biệt với bài **I** (LeetCode 1347: chỉ được **thay** ký tự trong `t`, hai xâu cùng độ dài).
5. Kiểm thử đối chiếu với mô phỏng tham lam và với vét cạn BFS trên xâu rất ngắn.

## 2. Đề bài & ví dụ

Cho hai xâu `s` và `t`. Một bước: **thêm** một ký tự bất kỳ vào cuối `s` hoặc `t`. Trả về số bước ít nhất để `s` và `t` là đảo chữ của nhau.

**Ví dụ 1:** `s = "leetcode"`, `t = "coats"` → `7` (thêm `"as"` vào `s`, thêm `"leede"` vào `t`).

**Ví dụ 2:** `s = "night"`, `t = "thing"` → `0`.

**Ràng buộc:** `1 ≤ len(s), len(t) ≤ 2·10⁵`, chỉ chữ thường.

In [1]:
import random
import time
from collections import Counter, deque
from typing import List

examples = [("leetcode", "coats", 7), ("night", "thing", 0)]

for s, t, exp in examples:
    cs, ct = Counter(s), Counter(t)
    print(f"s={s!r}: {dict(sorted(cs.items()))}")
    print(f"t={t!r}: {dict(sorted(ct.items()))}   kỳ vọng {exp}")

s='leetcode': {'c': 1, 'd': 1, 'e': 3, 'l': 1, 'o': 1, 't': 1}
t='coats': {'a': 1, 'c': 1, 'o': 1, 's': 1, 't': 1}   kỳ vọng 7
s='night': {'g': 1, 'h': 1, 'i': 1, 'n': 1, 't': 1}
t='thing': {'g': 1, 'h': 1, 'i': 1, 'n': 1, 't': 1}   kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Anagram ⇔ `Counter` bằng nhau ⇔ `sorted` bằng nhau

In [2]:
print(Counter("night") == Counter("thing"), sorted("night") == sorted("thing"), Counter("ab") == Counter("abb"))

True True False


### 3.2 Phép toán trên `Counter`

- `a − b`: giữ phần **dương** của hiệu (phần `a` có nhiều hơn `b`);
- `a & b`: giao đa tập (lấy `min`); `a | b`: hợp (lấy `max`).

In [3]:
a, b = Counter("leetcode"), Counter("coats")
print("s - t:", dict(a - b), "| t - s:", dict(b - a))
print("s & t:", dict(a & b), "| s | t:", dict(a | b))

s - t: {'l': 1, 'e': 3, 'd': 1} | t - s: {'a': 1, 's': 1}
s & t: {'t': 1, 'c': 1, 'o': 1} | s | t: {'l': 1, 'e': 3, 't': 1, 'c': 1, 'o': 1, 'd': 1, 'a': 1, 's': 1}


### 3.3 Mảng chênh lệch 26 ô

Một mảng `d[c] = cnt_s[c] − cnt_t[c]`: tăng khi đọc `s`, giảm khi đọc `t`; đáp án `Σ |d[c]|`.

In [4]:
d = [0] * 26
for ch in "leetcode": d[ord(ch) - 97] += 1
for ch in "coats":    d[ord(ch) - 97] -= 1
print({chr(97 + i): v for i, v in enumerate(d) if v})

{'a': -1, 'd': 1, 'e': 3, 'l': 1, 's': -1}


## 4. Ý tưởng nền tảng / trực giác

Chỉ được **thêm** ⇒ số lần xuất hiện của mỗi chữ cái chỉ **tăng**. Cuối cùng hai xâu phải có cùng số lần cho mỗi chữ `c`, tức ít nhất `max(cnt_s[c], cnt_t[c])` bản ở **mỗi** xâu. Rẻ nhất là đúng bằng `max`: xâu thiếu phải thêm `|cnt_s[c] − cnt_t[c]|` bản `c`.

Tính tay ví dụ 1:

| chữ | `s` | `t` | chênh |
|---|---|---|---|
| `a` | 0 | 1 | 1 |
| `c` | 1 | 1 | 0 |
| `d` | 1 | 0 | 1 |
| `e` | 3 | 0 | 3 |
| `l` | 1 | 0 | 1 |
| `o` | 1 | 1 | 0 |
| `s` | 0 | 1 | 1 |
| `t` | 1 | 1 | 0 |

Tổng chênh `1 + 1 + 3 + 1 + 1 = 7` ✓ (thêm `a, s` vào `s` — 2 bước; thêm `d, e, e, e, l` vào `t` — 5 bước).

## 5. Cách 1 — Brute Force / baseline: mô phỏng tham lam thêm từng ký tự

Lặp: tìm một chữ cái mà một bên có ít hơn bên kia, thêm **một** bản vào bên thiếu, đếm bước. Dừng khi hai đa tập bằng nhau. Mô phỏng đúng thao tác của đề (rất chậm khi chênh lệch lớn, nhưng dễ tin).

In [5]:
def steps_simulate(s, t):
    cs, ct = Counter(s), Counter(t)
    steps = 0
    while cs != ct:
        for c in set(cs) | set(ct):
            if cs[c] < ct[c]:
                cs[c] += 1; steps += 1; break
            if ct[c] < cs[c]:
                ct[c] += 1; steps += 1; break
    return steps

for s, t, exp in examples:
    print(steps_simulate(s, t), "(kỳ vọng", exp, ")")

7 (kỳ vọng 7 )
0 (kỳ vọng 0 )


**Độ phức tạp:** `O(D · σ)` với `D` = đáp án (mỗi bước quét các chữ). Với xâu `2·10⁵` khác hẳn nhau, `D` tới `4·10⁵` ⇒ chậm nhưng vẫn là baseline. Còn một baseline "thật sự vét cạn" (BFS trên mọi trạng thái) ở mục 12 cho xâu cực ngắn — để chứng minh tham lam trên là tối ưu.

**Nút thắt:** từng bước một. Ta biết luôn số bước của mỗi chữ cái là `|chênh|` ⇒ cộng thẳng.

## 6. Ý tưởng thuật toán / bước đột phá

**Cận dưới:** sau mọi dãy thao tác, với mỗi `c`, số `c` cuối cùng ở hai xâu bằng nhau `= f_c ≥ max(cnt_s[c], cnt_t[c])`. Số bước `= Σ_c (f_c − cnt_s[c]) + (f_c − cnt_t[c]) ≥ Σ_c |cnt_s[c] − cnt_t[c]|`.

**Đạt được:** chọn `f_c = max(...)` cho mọi `c`.

**Công thức qua giao đa tập:** `|x − y| = x + y − 2·min(x, y)` ⇒ đáp án `= len(s) + len(t) − 2·|s ∩ t|` với `|s ∩ t| = Σ_c min(cnt_s[c], cnt_t[c])`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `Counter` hai chiều

In [6]:
def steps_counter(s, t):
    cs, ct = Counter(s), Counter(t)
    return sum((cs - ct).values()) + sum((ct - cs).values())

for s, t, exp in examples:
    print(steps_counter(s, t), "(kỳ vọng", exp, ")")

7 (kỳ vọng 7 )
0 (kỳ vọng 0 )


**Độ phức tạp:** `O(|s| + |t|)`; `cs − ct` chỉ giữ phần dương nên cần **cả hai** chiều.

### 7.2 Cách 3 — Mảng chênh lệch 26 ô

In [7]:
def steps_array(s, t):
    d = [0] * 26
    for ch in s:
        d[ord(ch) - 97] += 1
    for ch in t:
        d[ord(ch) - 97] -= 1
    return sum(abs(x) for x in d)

for s, t, exp in examples:
    print(steps_array(s, t), "(kỳ vọng", exp, ")")

7 (kỳ vọng 7 )
0 (kỳ vọng 0 )


**Độ phức tạp:** `O(|s| + |t| + 26)`, bộ nhớ `O(26)`.

### 7.3 Cách 4 — Sắp xếp + trộn hai con trỏ

Sắp `s` và `t`; trộn như merge sort: hai ký tự bằng nhau ⇒ "khớp" (không tốn bước), tiến cả hai; khác nhau ⇒ ký tự nhỏ hơn không có cặp ⇒ tốn 1 bước, tiến con trỏ đó. Phần còn lại cuối mỗi xâu đều tốn bước.

In [8]:
def steps_sort_merge(s, t):
    a, b = sorted(s), sorted(t)
    i = j = steps = 0
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            i += 1; j += 1
        elif a[i] < b[j]:
            steps += 1; i += 1
        else:
            steps += 1; j += 1
    return steps + (len(a) - i) + (len(b) - j)

for s, t, exp in examples:
    print(steps_sort_merge(s, t), "(kỳ vọng", exp, ")")

7 (kỳ vọng 7 )
0 (kỳ vọng 0 )


**Độ phức tạp:** `O(n log n)` — chậm hơn đếm nhưng không cần biết bảng chữ cái nhỏ (dùng được cho Unicode, số nguyên bất kỳ).

### 7.4 Cách 5 — Công thức giao đa tập

In [9]:
def steps_intersection(s, t):
    common = sum((Counter(s) & Counter(t)).values())
    return len(s) + len(t) - 2 * common

for s, t, exp in examples:
    print(steps_intersection(s, t), "(kỳ vọng", exp, ")")

7 (kỳ vọng 7 )
0 (kỳ vọng 0 )


**Độ phức tạp:** `O(|s| + |t|)`. Cách nhìn "giữ lại phần chung, bù phần riêng" giúp nhận ra liên hệ với bài **LCS/edit distance** chỉ có thao tác thêm (nhưng ở đây thứ tự không quan trọng nên chỉ cần đa tập).

## 8. TRACE từng bước

### 8.1 Trace cách 3 trên ví dụ 1

In [10]:
def trace_array(s, t):
    d = [0] * 26
    for ch in s: d[ord(ch) - 97] += 1
    print("  sau khi đọc s:", {chr(97 + i): v for i, v in enumerate(d) if v})
    for ch in t: d[ord(ch) - 97] -= 1
    print("  sau khi trừ t:", {chr(97 + i): v for i, v in enumerate(d) if v})
    add_s = "".join(chr(97 + i) * -v for i, v in enumerate(d) if v < 0)
    add_t = "".join(chr(97 + i) * v for i, v in enumerate(d) if v > 0)
    print(f"  thêm vào s: {add_s!r} ({len(add_s)} bước), thêm vào t: {add_t!r} ({len(add_t)} bước)")
    print(f"  kiểm: Counter(s + add_s) == Counter(t + add_t)? {Counter(s + add_s) == Counter(t + add_t)}")
    return len(add_s) + len(add_t)

print("=>", trace_array("leetcode", "coats"))

  sau khi đọc s: {'c': 1, 'd': 1, 'e': 3, 'l': 1, 'o': 1, 't': 1}
  sau khi trừ t: {'a': -1, 'd': 1, 'e': 3, 'l': 1, 's': -1}
  thêm vào s: 'as' (2 bước), thêm vào t: 'deeel' (5 bước)
  kiểm: Counter(s + add_s) == Counter(t + add_t)? True
=> 7


### 8.2 Trace cách 4 (trộn) trên `s = "aabcc"`, `t = "abbd"` (ví dụ tự dựng — đã đối chiếu mô phỏng, đối chiếu lại đề gốc)

In [11]:
def trace_merge(s, t):
    a, b = sorted(s), sorted(t); i = j = steps = 0
    print(f"  a = {a}, b = {b}")
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            print(f"    a[{i}]={a[i]} == b[{j}]={b[j]} -> khớp"); i += 1; j += 1
        elif a[i] < b[j]:
            steps += 1; print(f"    a[{i}]={a[i]} < b[{j}]={b[j]} -> '{a[i]}' thiếu ở t, +1 (tổng {steps})"); i += 1
        else:
            steps += 1; print(f"    a[{i}]={a[i]} > b[{j}]={b[j]} -> '{b[j]}' thiếu ở s, +1 (tổng {steps})"); j += 1
    rest = (len(a) - i) + (len(b) - j)
    print(f"    phần còn lại: {a[i:]} + {b[j:]} -> +{rest}")
    return steps + rest

r = trace_merge("aabcc", "abbd")
print("  =>", r, "| mô phỏng:", steps_simulate("aabcc", "abbd"))

  a = ['a', 'a', 'b', 'c', 'c'], b = ['a', 'b', 'b', 'd']
    a[0]=a == b[0]=a -> khớp
    a[1]=a < b[1]=b -> 'a' thiếu ở t, +1 (tổng 1)
    a[2]=b == b[1]=b -> khớp
    a[3]=c > b[2]=b -> 'b' thiếu ở s, +1 (tổng 2)
    a[3]=c < b[3]=d -> 'c' thiếu ở t, +1 (tổng 3)
    a[4]=c < b[3]=d -> 'c' thiếu ở t, +1 (tổng 4)
    phần còn lại: [] + ['d'] -> +1
  => 5 | mô phỏng: 5


## 9. Code nộp LeetCode

In [12]:
class Solution:
    def minSteps(self, s: str, t: str) -> int:
        diff = [0] * 26
        for ch in s:
            diff[ord(ch) - 97] += 1
        for ch in t:
            diff[ord(ch) - 97] -= 1
        return sum(abs(x) for x in diff)      # mỗi chữ cái: bù phần chênh lệch

for s, t, exp in examples:
    assert Solution().minSteps(s, t) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Cận dưới.** Gọi `f_c` là số lần chữ `c` trong hai xâu **sau khi** hoàn tất (bằng nhau vì là đảo chữ). Do chỉ thêm, `f_c ≥ cnt_s[c]` và `f_c ≥ cnt_t[c]`. Số bước thêm vào `s` là `Σ (f_c − cnt_s[c])`, vào `t` là `Σ (f_c − cnt_t[c])`. Tổng `= Σ_c (2 f_c − cnt_s[c] − cnt_t[c]) ≥ Σ_c (2 max − cnt_s − cnt_t) = Σ_c |cnt_s[c] − cnt_t[c]|`.

**Đạt được.** Với `f_c = max(cnt_s[c], cnt_t[c])`: thêm `max − cnt_s[c]` bản `c` vào `s`, `max − cnt_t[c]` vào `t`; số bước đúng bằng cận dưới, và hai xâu có cùng đa tập ⇒ là đảo chữ.

**Cách 4** đếm đúng số ký tự không ghép cặp được trong đa tập (trộn hai dãy đã sắp ghép tối đa `min(cnt_s[c], cnt_t[c])` cặp cho mỗi `c`) — bằng `len(s) + len(t) − 2|s ∩ t|`.

## 11. Phân tích Time / Space Complexity

Gọi `n = |s|`, `m = |t|`, `σ = 26`.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng | `O(D · σ)` (`D` = đáp án) | `O(σ)` |
| 2. `Counter` hai chiều | `O(n + m)` | `O(σ)` |
| 3. Mảng chênh lệch | `O(n + m + σ)` | `O(σ)` |
| 4. Sắp + trộn | `O((n + m) log(n + m))` | `O(n + m)` |
| 5. Giao đa tập | `O(n + m)` | `O(σ)` |

In [13]:
rnd = random.Random(2186)
s_big = "".join(rnd.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(2 * 10**5))
t_big = "".join(rnd.choice("abcdefghijklm") for _ in range(2 * 10**5))
for name, fn in [("counter", steps_counter), ("array", steps_array), ("sort_merge", steps_sort_merge),
                 ("intersection", steps_intersection), ("Solution", Solution().minSteps)]:
    t0 = time.perf_counter(); r = fn(s_big, t_big); dt = time.perf_counter() - t0
    print(f"{name:<13}: {r}  {dt * 1000:7.1f} ms")

counter      : 200100      9.1 ms
array        : 200100     13.5 ms
sort_merge   : 200100     53.9 ms
intersection : 200100      9.4 ms


Solution     : 200100     14.0 ms


## 12. Bộ test

In [14]:
def steps_bfs(s, t, alphabet):
    """Vét cạn thật sự: BFS trên trạng thái (đa tập s, đa tập t) — chỉ cho xâu rất ngắn và bảng chữ nhỏ."""
    start = (tuple(Counter(s)[c] for c in alphabet), tuple(Counter(t)[c] for c in alphabet))
    q = deque([(start, 0)]); seen = {start}
    cap = len(s) + len(t)
    while q:
        (a, b), d = q.popleft()
        if a == b:
            return d
        for i in range(len(alphabet)):
            for side in (0, 1):
                na, nb = list(a), list(b)
                (na if side == 0 else nb)[i] += 1
                st = (tuple(na), tuple(nb))
                if max(max(na), max(nb)) <= cap and st not in seen:
                    seen.add(st); q.append((st, d + 1))

ALL = {
    "counter": steps_counter,
    "array": steps_array,
    "sort_merge": steps_sort_merge,
    "intersection": steps_intersection,
    "Solution": lambda s, t: Solution().minSteps(s, t),
}

def check(s, t, exp=None, ref_fn=steps_simulate):
    ref = ref_fn(s, t)
    for name, fn in ALL.items():
        got = fn(s, t)
        assert got == ref, (name, s, t, got, ref)
    if exp is not None:
        assert ref == exp
    return ref

for s_, t_, exp in examples:
    check(s_, t_, exp)
print("Ví dụ đề: OK")

check("a", "a", 0); check("a", "b", 2); check("aaa", "a", 2)
check("zzzz", "zz", 2); check("abcdefghijklmnopqrstuvwxyz", "zyxwvutsrqponmlkjihgfedcba", 0)
check("ab", "ba", 0); check("aabcc", "abbd")
print("Edge case: OK")

rnd = random.Random(2021)
for _ in range(150):
    s_ = "".join(rnd.choice("ab") for _ in range(rnd.randint(1, 3)))
    t_ = "".join(rnd.choice("ab") for _ in range(rnd.randint(1, 3)))
    assert steps_bfs(s_, t_, "ab") == steps_simulate(s_, t_) == Solution().minSteps(s_, t_)
print("BFS vét cạn (bảng chữ {a,b}, độ dài ≤ 3) khẳng định mô phỏng tham lam là tối ưu: OK")

for _ in range(2000):
    s_ = "".join(rnd.choice("abcde") for _ in range(rnd.randint(1, 15)))
    t_ = "".join(rnd.choice("abcxy") for _ in range(rnd.randint(1, 15)))
    check(s_, t_)
print("2000 cặp xâu ngẫu nhiên so với mô phỏng: OK")

for _ in range(5):
    n, m = rnd.randint(1, 2 * 10**5), rnd.randint(1, 2 * 10**5)
    s_ = "".join(rnd.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(n))
    t_ = "".join(rnd.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(m))
    check(s_, t_, ref_fn=steps_array)
print("5 cặp xâu lớn (đối chiếu chéo): OK")

Ví dụ đề: OK
Edge case: OK
BFS vét cạn (bảng chữ {a,b}, độ dài ≤ 3) khẳng định mô phỏng tham lam là tối ưu: OK


2000 cặp xâu ngẫu nhiên so với mô phỏng: OK


5 cặp xâu lớn (đối chiếu chéo): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ đếm một chiều `Counter(s) − Counter(t)`

In [15]:
def bug_one_side(s, t):
    return sum((Counter(s) - Counter(t)).values())

print("sai :", bug_one_side("leetcode", "coats"), "| đúng:", Solution().minSteps("leetcode", "coats"))

sai : 5 | đúng: 7


**Vì sao sai:** chỉ đếm phần `s` thừa (phải thêm vào `t`), quên phần `t` thừa (`a, s` phải thêm vào `s`). **Sửa:** cộng cả hai chiều, hoặc `Σ |d[c]|`.

### Lỗi 2 — Nhầm với bài I: "thay ký tự" ⇒ chỉ lấy một chiều (hoặc chia đôi)

In [16]:
def bug_half(s, t):
    d = [0] * 26
    for ch in s: d[ord(ch) - 97] += 1
    for ch in t: d[ord(ch) - 97] -= 1
    return sum(abs(x) for x in d) // 2              # SAI: đó là công thức khi được THAY ký tự và |s| = |t|

print("sai :", bug_half("leetcode", "coats"), "| đúng:", Solution().minSteps("leetcode", "coats"))

sai : 3 | đúng: 7


**Vì sao sai:** ở bài 1347, một lần **thay** sửa được hai đơn vị chênh lệch cùng lúc; ở bài này mỗi bước **thêm** chỉ sửa một đơn vị. **Sửa:** không chia đôi.

### Lỗi 3 — Đếm số **chữ cái khác nhau** bị lệch thay vì tổng độ lệch

In [17]:
def bug_distinct(s, t):
    cs, ct = Counter(s), Counter(t)
    return sum(1 for c in set(cs) | set(ct) if cs[c] != ct[c])

print("sai :", bug_distinct("leetcode", "coats"), "| đúng:", Solution().minSteps("leetcode", "coats"))

sai : 5 | đúng: 7


**Vì sao sai:** chữ `e` lệch 3 nhưng chỉ được đếm 1. **Sửa:** cộng `|cnt_s − cnt_t|`.

### Lỗi 4 — Dùng `set` (bỏ qua số lần xuất hiện)

In [18]:
def bug_set(s, t):
    return len(set(s) ^ set(t))

print("sai :", bug_set("leetcode", "coats"), "| đúng:", Solution().minSteps("leetcode", "coats"))
print("sai :", bug_set("aaa", "a"), "| đúng:", Solution().minSteps("aaa", "a"))

sai : 5 | đúng: 7
sai : 0 | đúng: 2


**Vì sao sai:** đảo chữ cần cùng **đa tập**, không chỉ cùng tập chữ cái. `"aaa"` và `"a"` có cùng tập nhưng cần thêm 2 `a`.

### Lỗi 5 — Trả về hiệu độ dài `|len(s) − len(t)|`

In [19]:
def bug_length(s, t):
    return abs(len(s) - len(t))

print("sai :", bug_length("ab", "cd"), "| đúng:", Solution().minSteps("ab", "cd"))

sai : 0 | đúng: 4


**Vì sao sai:** đây chỉ là **cận dưới** rất lỏng (hai xâu cùng độ dài vẫn có thể khác hẳn nhau). **Sửa:** so từng chữ cái.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đảo chữ" ⇒ so **đa tập** ⇒ `Counter` / mảng 26 ô.
- Chỉ được **thêm** ⇒ mỗi chữ cái tốn `|chênh|`; được **thay** (cùng độ dài) ⇒ tốn `Σ max(0, chênh)` = một nửa tổng `|chênh|`; được **xóa** ⇒ cũng `|chênh|` (đối xứng với thêm).
- Một mảng chênh lệch duy nhất (`+` cho `s`, `−` cho `t`) gọn hơn hai bộ đếm.
- `|x − y| = x + y − 2 min(x, y)` ⇒ công thức qua giao đa tập.
- Bảng chữ cái lớn/không biết trước ⇒ sắp + trộn hoặc `Counter`.

## 15. Biến thể của bài

1. **LeetCode 1347 — Minimum Number of Steps to Make Two Strings Anagram (I):** cùng độ dài, chỉ **thay** ký tự trong `t` ⇒ `Σ max(0, cnt_s − cnt_t)`.
2. **LeetCode 242 — Valid Anagram:** kiểm tra bằng nhau.
3. **LeetCode 438 — Find All Anagrams in a String:** cửa sổ trượt + mảng chênh lệch.
4. **LeetCode 1657 — Determine if Two Strings Are Close.**
5. **Chỉ được xóa** ký tự ở cả hai xâu ⇒ cũng `Σ |chênh|` (xóa phần thừa) — đối ngẫu của bài này.

In [20]:
def steps_replace_1347(s, t):
    """Bài I: |s| = |t|, mỗi bước THAY một ký tự trong t."""
    d = [0] * 26
    for ch in s: d[ord(ch) - 97] += 1
    for ch in t: d[ord(ch) - 97] -= 1
    return sum(x for x in d if x > 0)

def find_anagrams_438(s, p):
    """Bài 438: vị trí bắt đầu các cửa sổ là anagram của p (mảng chênh lệch + đếm số ô khác 0)."""
    if len(p) > len(s):
        return []
    d = [0] * 26
    for ch in p: d[ord(ch) - 97] -= 1
    nonzero = sum(1 for x in d if x)
    out = []
    def bump(i, v):
        nonlocal nonzero
        before = d[i] != 0; d[i] += v; after = d[i] != 0
        nonzero += after - before
    for i, ch in enumerate(s):
        bump(ord(ch) - 97, 1)
        if i >= len(p):
            bump(ord(s[i - len(p)]) - 97, -1)
        if i >= len(p) - 1 and nonzero == 0:
            out.append(i - len(p) + 1)
    return out

print("1347 ('bab','aba') ->", steps_replace_1347("bab", "aba"), "| ('leetcode','practice') ->", steps_replace_1347("leetcode", "practice"))
print("438 ('cbaebabacd','abc') ->", find_anagrams_438("cbaebabacd", "abc"))
rnd_v = random.Random(15)
for _ in range(500):
    s_ = "".join(rnd_v.choice("abc") for _ in range(rnd_v.randint(1, 12))); p_ = "".join(rnd_v.choice("abc") for _ in range(rnd_v.randint(1, 4)))
    assert find_anagrams_438(s_, p_) == [i for i in range(len(s_) - len(p_) + 1) if Counter(s_[i:i + len(p_)]) == Counter(p_)]
print("438 khớp vét cạn trên 500 cặp")

1347 ('bab','aba') -> 1 | ('leetcode','practice') -> 5
438 ('cbaebabacd','abc') -> [0, 6]
438 khớp vét cạn trên 500 cặp


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | thêm từng ký tự | `O(Dσ)` | `O(σ)` | baseline |
| 2. `Counter` 2 chiều | `(cs−ct) + (ct−cs)` | `O(n+m)` | `O(σ)` | Python gọn |
| 3. Mảng chênh lệch | `Σ|d|` | `O(n+m)` | `O(26)` | **nộp bài** |
| 4. Sắp + trộn | ghép cặp | `O(N log N)` | `O(N)` | bảng chữ lớn |
| 5. Giao đa tập | `n+m−2|∩|` | `O(n+m)` | `O(σ)` | công thức đẹp |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minSteps("aabb", "abcc")`?

**Bài 2 (tính tay):** Với `s = "abc"`, `t = "aabbcd"`, liệt kê ký tự cần thêm vào mỗi xâu.

**Bài 3 (điền TODO):** Hoàn thiện `make_anagrams(s, t)` — trả về **cặp xâu** `(s', t')` sau khi thêm tối thiểu (thêm vào cuối, các ký tự thêm sắp tăng dần).

**Bài 4 (tự viết lại):** Viết lại cách 3 với **một vòng lặp** duyệt đồng thời hai xâu bằng `itertools.zip_longest`.

**Bài 5 (mini challenge):** Mỗi bước thêm chữ `c` có **chi phí** `cost[c]` (khác nhau theo chữ). Tìm chi phí nhỏ nhất. Rồi biến thể khó hơn: được **thêm** (chi phí `add[c]`) hoặc **xóa** (chi phí `rem[c]`) ở bất kỳ xâu nào — tìm chi phí nhỏ nhất (gợi ý: với mỗi chữ `c` độc lập, chọn mức cuối `f_c` tối ưu trong `[min, max]`). Kiểm bằng vét cạn `f_c`.

In [21]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def make_anagrams(s, t):
    # TODO: d[c] < 0 -> thêm vào s; d[c] > 0 -> thêm vào t
    pass

# Bài 4 — TODO
def steps_zip(s, t):
    # TODO: zip_longest(s, t, fillvalue=None), cập nhật d trong một vòng
    pass

# Bài 5 — TODO
def min_cost_add(s, t, cost):
    # TODO: Σ |d[c]| * cost[c]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `s`: `a2 b2`; `t`: `a1 b1 c2`.
- Mức 2: Chênh `a:1, b:1, c:2`.
- Mức 3: `4`.

**Bài 2**
- Mức 1: `t` có thêm `a, b, d`.
- Mức 2: Thêm `a, b, d` vào `s`.
- Mức 3: Không cần thêm gì vào `t`; 3 bước.

**Bài 3**
- Mức 1: Dùng mảng chênh `d`.
- Mức 2: `s' = s + "".join(c * (−d[c]) ...)`.
- Mức 3: Kiểm `Counter(s') == Counter(t')` và `len(s') + len(t') − len(s) − len(t) == ` đáp án.

**Bài 4**
- Mức 1: `zip_longest` trả `None` khi một xâu hết.
- Mức 2: `if a: d[a] += 1; if b: d[b] −= 1`.
- Mức 3: So với `Solution`.

**Bài 5**
- Mức 1: Chi phí thêm độc lập theo chữ: `Σ |d[c]| · cost[c]`.
- Mức 2: Có cả xóa: mức cuối `f_c` giữa `lo = min(x, y)` và `hi = max(x, y)`; chi phí `(f−x)⁺·add + (x−f)⁺·rem + (f−y)⁺·add + (y−f)⁺·rem`.
- Mức 3: Hàm tuyến tính từng khúc theo `f` ⇒ tối ưu tại `lo` hoặc `hi` (hoặc ngoài khoảng? — không, ra ngoài chỉ tốn thêm). Đối chiếu vét cạn `f ∈ [0, x + y]`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
from itertools import zip_longest

# Bài 1
r1 = Solution().minSteps("aabb", "abcc")
print("Bài 1:", r1); assert r1 == 4

# Bài 2
d2 = Counter("aabbcd") - Counter("abc"); e2 = Counter("abc") - Counter("aabbcd")
print("Bài 2: thêm vào s:", "".join(sorted(d2.elements())), "| thêm vào t:", "".join(sorted(e2.elements())))
assert "".join(sorted(d2.elements())) == "abd" and not e2

# Bài 3
def make_anagrams_sol(s, t):
    d = [0] * 26
    for ch in s: d[ord(ch) - 97] += 1
    for ch in t: d[ord(ch) - 97] -= 1
    add_s = "".join(chr(97 + i) * -v for i, v in enumerate(d) if v < 0)
    add_t = "".join(chr(97 + i) * v for i, v in enumerate(d) if v > 0)
    return s + add_s, t + add_t

rnd = random.Random(3)
for _ in range(1000):
    s_ = "".join(rnd.choice("abcd") for _ in range(rnd.randint(1, 10))); t_ = "".join(rnd.choice("abce") for _ in range(rnd.randint(1, 10)))
    a, b = make_anagrams_sol(s_, t_)
    assert a.startswith(s_) and b.startswith(t_) and Counter(a) == Counter(b)
    assert len(a) + len(b) - len(s_) - len(t_) == Solution().minSteps(s_, t_)
print("Bài 3 OK, ví dụ:", make_anagrams_sol("leetcode", "coats"))

Bài 1: 4
Bài 2: thêm vào s: abd | thêm vào t: 
Bài 3 OK, ví dụ: ('leetcodeas', 'coatsdeeel')


In [23]:
# Bài 4
def steps_zip_sol(s, t):
    d = [0] * 26
    for a, b in zip_longest(s, t):
        if a: d[ord(a) - 97] += 1
        if b: d[ord(b) - 97] -= 1
    return sum(map(abs, d))

# Bài 5
def min_cost_add_sol(s, t, cost):
    cs, ct = Counter(s), Counter(t)
    return sum(abs(cs[c] - ct[c]) * cost[c] for c in set(cs) | set(ct))

def min_cost_add_remove(s, t, add, rem):
    cs, ct = Counter(s), Counter(t); total = 0
    for c in set(cs) | set(ct):
        x, y = cs[c], ct[c]
        f_cost = lambda f: max(0, f - x) * add[c] + max(0, x - f) * rem[c] + max(0, f - y) * add[c] + max(0, y - f) * rem[c]
        total += min(f_cost(min(x, y)), f_cost(max(x, y)))
    return total

def add_remove_brute(s, t, add, rem):
    cs, ct = Counter(s), Counter(t); total = 0
    for c in set(cs) | set(ct):
        x, y = cs[c], ct[c]
        total += min(max(0, f - x) * add[c] + max(0, x - f) * rem[c] + max(0, f - y) * add[c] + max(0, y - f) * rem[c]
                     for f in range(0, x + y + 1))
    return total

rnd = random.Random(5)
for _ in range(1000):
    s_ = "".join(rnd.choice("abc") for _ in range(rnd.randint(1, 10))); t_ = "".join(rnd.choice("abd") for _ in range(rnd.randint(1, 10)))
    assert steps_zip_sol(s_, t_) == Solution().minSteps(s_, t_)
    add = {c: rnd.randint(1, 9) for c in "abcd"}; rem = {c: rnd.randint(1, 9) for c in "abcd"}
    assert min_cost_add_sol(s_, t_, {c: 1 for c in "abcd"}) == Solution().minSteps(s_, t_)
    assert min_cost_add_remove(s_, t_, add, rem) == add_remove_brute(s_, t_, add, rem)
print("Bài 4, Bài 5 OK; ví dụ chi phí thêm/xóa:", min_cost_add_remove("aaab", "b", {"a": 5, "b": 1}, {"a": 1, "b": 1}))

Bài 4, Bài 5 OK; ví dụ chi phí thêm/xóa: 3


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
d = [0] * 26
for ch in s: d[ord(ch) - 97] += 1
for ch in t: d[ord(ch) - 97] -= 1
return sum(abs(x) for x in d)
```

**Checklist:**
- [ ] Cộng **cả hai** chiều chênh lệch?
- [ ] Không chia đôi (đó là bài "thay ký tự")?
- [ ] Đếm theo **số lần**, không theo tập chữ cái?
- [ ] Xâu độ dài khác nhau vẫn đúng?
- [ ] Chữ ký đúng: `def minSteps(self, s: str, t: str) -> int`?